# RailGuard Vision — Phase 2: YOLOv8 Training & Validation (Colab GPU)

**Project:** RailGuard Vision — YOLOv8 Railway Track Defect Detection System  
**Objective:** Train, validate, evaluate, and export fine-tuned YOLOv8 model weights for local deployment.

---

## Step 1: Install Ultralytics & Verify GPU Acceleration

In [ ]:
# Install Ultralytics YOLOv8 library
!pip install -q ultralytics matplotlib opencv-python pandas

import torch
from ultralytics import YOLO

# Verify GPU hardware availability
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU hardware not detected! Enable GPU runtime in Colab via Runtime -> Change runtime type.')

## Step 2: Initialize YOLOv8 Architecture & Hyperparameter Rationale

### Hyperparameter Selections:
- **Base Architecture (`yolov8s.pt`)**: YOLOv8 Small model provides an optimal tradeoff between computational efficiency (high FPS for video processing) and feature extraction capacity for small micro-cracks.
- **Epochs (80 - 100)**: Ensures adequate convergence while preventing overfitting.
- **Patience (15 epochs)**: Early stopping triggers if validation loss plateaus for 15 consecutive epochs.
- **Image Size (`imgsz=640`)**: Standard 640x640 resolution preserves spatial detail needed to detect narrow track fissures.
- **Batch Size (`batch=16`)**: Fits within typical 16GB GPU RAM limits while stabilizing AdamW/SGD gradient estimation.

In [ ]:
# Initialize pretrained YOLOv8 model checkpoint
model = YOLO('yolov8s.pt')

# Execute Model Training on data.yaml
results = model.train(
    data='./data/data.yaml',
    epochs=90,
    patience=15,
    imgsz=640,
    batch=16,
    project='RailGuard_YOLOv8',
    name='rail_defect_run',
    save=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    plots=True
)

print('YOLOv8 Model Training Run Successfully Completed!')

## Step 3: Plot & Analyze Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_csv = Path('./RailGuard_YOLOv8/rail_defect_run/results.csv')

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Box Loss Plot
    axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='blue')
    axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='red')
    axes[0, 0].set_title('Bounding Box Loss')
    axes[0, 0].grid(True)
    axes[0, 0].legend()
    
    # Class Loss Plot
    axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train Class Loss', color='blue')
    axes[0, 1].plot(df['epoch'], df['val/cls_loss'], label='Val Class Loss', color='red')
    axes[0, 1].set_title('Classification Loss')
    axes[0, 1].grid(True)
    axes[0, 1].legend()
    
    # mAP Metrics Plot
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5', color='green')
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='purple')
    axes[1, 0].set_title('Mean Average Precision (mAP)')
    axes[1, 0].grid(True)
    axes[1, 0].legend()
    
    # Precision & Recall Plot
    axes[1, 1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='orange')
    axes[1, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='teal')
    axes[1, 1].set_title('Precision & Recall')
    axes[1, 1].grid(True)
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print('results.csv file not found.')

## Step 4: Validation Run & Confusion Matrix

In [ ]:
from PIL import Image

# Run Validation on Held-Out Validation Dataset
best_model_path = './RailGuard_YOLOv8/rail_defect_run/weights/best.pt'
val_model = YOLO(best_model_path)
metrics = val_model.val()

print(f'Overall mAP@0.5: {metrics.box.map50 * 100:.2f}%')
print(f'Overall mAP@0.5:0.95: {metrics.box.map * 100:.2f}%')
print(f'Overall Precision: {metrics.box.mp * 100:.2f}%')
print(f'Overall Recall: {metrics.box.mr * 100:.2f}%')

# Display Generated Confusion Matrix Graphic
cm_path = Path('./RailGuard_YOLOv8/rail_defect_run/confusion_matrix.png')
if cm_path.exists():
    display(Image.open(cm_path))

## Step 5: Test Set Inference & Visualizing Predicted Bounding Boxes

In [ ]:
import glob
import cv2

test_images = glob.glob('./data/dataset_split/test/images/*.jpg')[:5]

fig, axes = plt.subplots(1, len(test_images), figsize=(20, 5))
for i, img_p in enumerate(test_images):
    res = val_model(img_p)[0]
    annotated_bgr = res.plot()
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    
    axes[i].imshow(annotated_rgb)
    axes[i].set_title(f'Test Image #{i+1}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## Step 6: Export & Download `best.pt` Weights to Google Drive / Local VS Code

In [ ]:
from google.colab import files, drive
import shutil

# Option A: Direct Download to Browser
print('Downloading best.pt weight file to your local computer...')
files.download(best_model_path)

# Option B: Save to Google Drive Sync Folder
try:
    drive.mount('/content/drive')
    gdrive_dest = '/content/drive/MyDrive/RailGuard_Vision_Models/'
    os.makedirs(gdrive_dest, exist_ok=True)
    shutil.copy(best_model_path, os.path.join(gdrive_dest, 'best.pt'))
    print(f'Successfully copied best.pt to Google Drive: {gdrive_dest}')
except Exception as e:
    print(f'Google Drive sync skipped ({e}).')